In [13]:
import mlflow
import mlflow.transformers

import torch
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from pathlib import Path

In [14]:
llm_llama = "meta-llama/Llama-3.2-1B-Instruct"
 
llm_SmolLM2="HuggingFaceTB/SmolLM2-360M-Instruct"
max_tokens=30

#### Set up mlflow

In [15]:
project_root=Path("__file__").resolve().parents[1]


mlflow_db=project_root/"mlflow.db"
mlflow_uri=f"sqlite:///{mlflow_db.as_posix()}"

mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_registry_uri(mlflow_uri)

#### Load DistilBERT champion

In [16]:
sentiment_model=mlflow.transformers.load_model(
    "models:/financial_sentiment_distilbert@champion", 
    device=0)

label_encoder=joblib.load(
    project_root/"outputs/encoders/label_encoder.pkl"
)    


2026/09/10 09:46:44 INFO mlflow.transformers: 'models:/financial_sentiment_distilbert@champion' resolved as 'file:///C:/Users/Kun%20Bi/Desktop/financial_sentiments/mlartifacts/models/m-c20c6b4700084aa49298d5429e1feb40/artifacts'


2026/09/10 09:46:44 WARNING mlflow.transformers.model_io: Could not specify device parameter for this pipeline type.Falling back to loading the model with the default device.
Device set to use cuda:0


In [17]:
texts = [
    "The company reported record quarterly revenue and raised its full-year guidance.",
    "Shares fell sharply after management warned of weaker consumer demand.",
    "Revenue was largely unchanged from the previous quarter.",
    "The firm beat analysts' earnings expectations despite higher operating costs.",
    "Management expects margins to remain under pressure for the rest of the year.",
    "The company announced a new share buyback program worth $5 billion.",
    "Quarterly results were broadly in line with market expectations.",
    "Sales declined 12% as demand weakened across major markets.",
    "Free cash flow improved significantly and debt levels continued to fall.",
    "The company maintained its previous outlook for the fiscal year."
]

In [18]:
outputs=sentiment_model(texts, batch_size=10)

pred_ids=np.array([
    int(output["label"].split("_")[-1]) for output in outputs
])

pred_labels=label_encoder.inverse_transform(pred_ids)

pred_probabilities=np.array([
    output["score"] for output in outputs
])

#### Seting up prompt and explaination function

In [19]:
system_prompt="""
            You are a financial sentiment explanation assistant.
A machine learning classifier has already classified
the financial text as positive, neutral, or negative.

Your task is to explain the evidence in the text that
supports the prediction.

Use only information contained in the text.
Do not invent company information or outside facts.

If the evidence for the predicted sentiment is weak
or mixed, explicitly say so.
            """

In [20]:
def explain_sentiment(text, sentiment, confidence):
    messages=[
        {
            "role":"system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"Financial text:\n{text}\n\n"
                f"Predicted sentiment: {sentiment}\n"
                f"Classifier confidence: {confidence:.2%}\n\n"
                "Explain why the classifier may have assigned this "
                "sentiment. Identify the important financial signals "
                "or phrases. Keep the explanation to 2-3 sentences."
            )
        }
    ]

    inputs=llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():
        generated_ids=llm_model.generate(inputs, max_new_tokens=max_tokens, do_sample=False)
    
    generated_ids=generated_ids[:, inputs.shape[1]:]

    explanation=llm_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]
    return explanation

#### Load and track LLM in 4-bit

In [21]:
llm_model_id=llm_SmolLM2
quantization_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

mlflow.set_experiment("financial_sentiments")
with mlflow.start_run(run_name="qwen_sentiment_explainer"):
    mlflow.log_params({
        "llm_model":llm_model_id,
        "max_new_tokens":max_tokens,
        "do_sample": False,
        "taks": "sentiment_explanation"
    })

    mlflow.log_text(system_prompt,"prompts/system_prompt.txt")

    llm_tokenizer=AutoTokenizer.from_pretrained(llm_model_id)

    llm_model=AutoModelForCausalLM.from_pretrained(
        llm_model_id,
        quantization_config=quantization_config,
        device_map="auto"
    )

#### Generate Explaination for every prediction

In [22]:
explanations=[]
for text, sentiment, probability in zip(
    texts, pred_labels, pred_probabilities
):
    explanation=explain_sentiment(text=text,
    sentiment=sentiment,confidence=probability)

    explanations.append(explanation)

explain_sentiment

<function __main__.explain_sentiment(text, sentiment, confidence)>

In [23]:
df=pd.DataFrame({
    "Text": texts,
    "Sentiment": pred_labels,
    "Probability": [str(np.round(p*100,2))+"%" for p in pred_probabilities],
    "Explanation": [e.rsplit(".",1)[0]+"." for e in explanations if "." in e]
})


In [24]:
pd.set_option("display.max_colwidth", None)
df

,Text,Sentiment,Probability,Explanation
0,The company reported record quarterly revenue and raised its full-year guidance.,positive,89.71%,"The financial text suggests that the company has made significant progress in the past year, with record-breaking quarterly revenue and full-year guidance."
1,Shares fell sharply after management warned of weaker consumer demand.,negative,94.97%,"The financial text suggests that the financial sentiment of the company is negative, as the shares fell sharply after management warned of weaker consumer demand."
2,Revenue was largely unchanged from the previous quarter.,positive,57.87%,"The financial text suggests that revenue was largely unchanged from the previous quarter, which may indicate a stable market position."
3,The firm beat analysts' earnings expectations despite higher operating costs.,positive,91.93%,"The financial text suggests that the firm beat analysts' expectations, which is a positive sentiment."
4,Management expects margins to remain under pressure for the rest of the year.,neutral,45.56%,"The financial text suggests that the financial market may be experiencing a period of volatility, with the company facing potential challenges in the short-term."
5,The company announced a new share buyback program worth $5 billion.,positive,57.77%,"The financial text mentions a new share buyback program, which is a positive development for the company."
6,Quarterly results were broadly in line with market expectations.,positive,82.88%,"The financial text suggests that the financial sentiment analysis has been accurate, as the quarterly results were generally in line with market expectations."
7,Sales declined 12% as demand weakened across major markets.,negative,88.47%,"The financial text suggests that the decline in sales is primarily due to market weakness, which is a common phenomenon in the retail industry."
8,Free cash flow improved significantly and debt levels continued to fall.,neutral,43.0%,"The financial text suggests that the company's free cash flow improved significantly, which is a positive trend."
9,The company maintained its previous outlook for the fiscal year.,neutral,80.9%,"The financial text suggests that the company maintained its previous outlook for the fiscal year, indicating a stable and positive outlook."
